# IPSL-AID — Training Notebook

**Author:** Kishanthan Kingston  
**Institution:** IPSL / CNRS / Sorbonne University  
**Year:** 2026  

---

This notebook provides a reproducible example of how to:
- configure an experiment
- launch training

This notebook is intended for experimentation only.  
For large-scale or production runs, please use the SLURM pipeline.

In [1]:
# Environment setup (uv)
#
# To use the project's uv virtual environment in Jupyter Notebook:
#
# 1. Activate the environment:
#       source .venv/bin/activate
#
# 2. Register the environment as a Jupyter kernel:
#       python -m ipykernel install --user --name ipsl_aid --display-name "IPSL-AID (uv)"
#
# 3. In the notebook interface, select the kernel:
#       Kernel → Change Kernel → "IPSL-AID (uv)"
#
# The notebook will then run using the uv environment.

## **Command Description**

The following command builds and executes a full IPSL-AID experiment from the notebook.

It performs three main steps:

1. **Set working directory**  
   Moves to the project root directory so that all relative paths (data, configs, outputs, `.venv`) are resolved correctly.

2. **Activate the virtual environment**  
   Activates the local Python virtual environment to ensure the correct dependencies are used and the `ipsl-aid` CLI command is available.

3. **Run the training pipeline**  
   Launches the IPSL-AID pipeline via the `ipsl-aid` command with all required parameters:
   - experiment configuration (folders, naming)
   - model architecture and diffusion settings
   - dataset paths and variables
   - training hyperparameters

### **Model Types and Preconditioning**

IPSL-AID supports multiple model formulations through the `--precond` argument.  
This parameter fully defines the model formulation, including the architecture behavior and loss function.

#### **Available options:**

- **Diffusion models**
  - `--precond vp` → VP preconditioning + `VPLoss`
  - `--precond ve` → VE preconditioning + `VELoss`
  - `--precond edm` → EDM preconditioning + `EDMLoss` *(default)*

- **Direct U-Net**
  - `--precond unet` → standard U-Net + `UnetLoss`

  The architecture then depends on `--arch`:
  - `adm` → Dhariwal U-Net
  - `ddpmpp` or `ncsnpp` → Song U-Net

#### **Input channels configuration**

The definition of `in_channels` depends on the model type:

- **Diffusion models**
  - `in_channels = number of variables`
  - `cond_channels = number of variables + constants (lat, lon, lsm, z)`

- **U-Net**
  - `in_channels = variables + constants + conditioning`
  - `no cond_channels`

This is a key difference between diffusion and U-Net models and must be set correctly to avoid shape mismatches.

#### **Inference behavior**

- Diffusion models require:
  - `--inference_type sampler`

- U-Net uses:
  - `--inference_type direct`

These parameters must be consistent:
- Diffusion + `sampler`
- U-Net + `direct`

### **Parameter overview**

This setup includes several parameters that can be grouped into the following categories:

#### **Experiment configuration**
- `--main_folder`, `--sub_folder`, `--prefix`  
  Define the output structure and naming of the experiment.
- `--run_type`  
  Defines the execution mode (`train`, `train_regional`, `resume_train`, `inference`, `inference_regional`).

#### **Regional configuration**

You can specify the region in two ways:

- **Predefined regions**
  - `--region europe`
  - `--region asia`
  - `--region us`

- **Custom region**
  - `--region_center lat lon`  
    Center of the region (in degrees).
  - `--region_size lat_size lon_size`  
    Spatial extent of the region.


#### **Region size**

The region size determines how many spatial blocks are used:

- `144 × 360` → **1 block**
- `288 × 720` → **4 blocks**


#### **Checkpoint**
- `--save_model`  
  Enable/disable checkpoint saving.
- `--save_checkpoint_name`  
  Base name used for saving checkpoints.
- `--load_checkpoint_name`  
  Checkpoint file to load (for resume or inference).
- `--save_per_samples`  
  Frequency of checkpoint saving.


#### **Model configuration**
- `--arch`  
  Model architecture (`adm`, `ddpmpp`, `ncsnpp`).
- `--precond`  
  Defines the model type (diffusion or U-Net).
- `--in_channels`, `--cond_channels`, `--out_channels`  
  Define input/output tensor structure.
- `--inference_type`  
  `sampler` (diffusion) or `direct` (U-Net).


#### **Data configuration**
- `--datadir`  
  Path to dataset.
- `--per_var_datadir`  
  Per-variable data paths.
- `--varnames_list`  
  Variables used for training.
- `--constant_varnames_list`  
  Static variables (e.g. `z`, `lsm`).
- `--units_list`  
  Units corresponding to each variable.
- `--normalization_types`  
  Normalization method per variable.


#### **Time configuration**
- `--year_start`, `--year_end`  
  Training period.
- `--year_start_test`, `--year_end_test`  
  Validation period.
- `--time_normalization`  
  Encoding of temporal features (`linear`, `cos_sin`).


#### **Training parameters**
- `--batch_size`  
  Number of samples per batch.
- `--num_epochs`  
  Number of training epochs.
- `--learning_rate`  
  Optimizer learning rate.
- `--num_workers`  
  Data loading workers.
  

#### **Spatial & temporal batching**
- `--tbatch`  
  Temporal batch length.
- `--sbatch`  
  Number of spatial batches.
- `--batch_size_lat`, `--batch_size_lon`  
  Spatial patch size.


#### **Data processing**
- `--epsilon`, `--beta`, `--margin`  
  Parameters controlling filtering behavior.
- `--apply_filter`  
  Enable/disable filtering.


#### **Diffusion sampler (only for diffusion models)**
- `--num_steps`  
  Number of sampling steps.
- `--sigma_min`, `--sigma_max`  
  Noise schedule range.
- `--rho`  
  Time discretization parameter.
- `--s_churn`  
  Stochasticity strength.
- `--solver`  
  Numerical solver (`heun`, `euler`).


#### **Evaluation**
- `--compute_crps`  
  Enable probabilistic evaluation (diffusion only).

## Diffusion Model — Training Command

This example runs a **simple training configuration on a selected region**, with `--run_type` set to `train_regional`:
- **1 year of training data** (2019)
- **1 year of validation data** (2020)
- **1 epoch only**

It is intended for quick testing and validation of the pipeline.

### Outputs

All outputs are saved in the following directories:

- **Results (plots, diagnostics):**  
  `results/notebook_test/test_run_diffusion`

- **Logs:**  
  `logs/notebook_test/test_run_diffusion`

- **Checkpoints:**  
  `checkpoints/notebook_test/test_run_diffusion`

In [2]:
cmd = """

cd "/leonardo_work/EUHPC_D27_095/kkingston/IPSL-AID"
source .venv/bin/activate

ipsl-aid \
    --debug false \
    --main_folder notebook_test \
    --sub_folder test_run_diffusion_Europe \
    --prefix run_20260323_142528 \
    --run_type train_regional \
    --region "Europe" \
    --region_size 288 720 \
    --save_model true \
    --save_checkpoint_name difusion_model \
    --load_checkpoint_name difusion_model \
    --save_per_samples 10000 \
    --inference_type sampler \
    --arch adm \
    --precond edm \
    --in_channels 1 \
    --cond_channels 5 \
    --out_channels 1 \
    --apply_filter false \
    --model_name notebook_test_run_diffusion_Europe \
    --varnames_list VAR_2T \
    --normalization_types VAR_2T=standard \
    --constant_varnames_list z lsm \
    --constant_varnames_file ERA5_const_sfc_variables.nc \
    --units_list K \
    --year_start 2019 \
    --year_end 2019 \
    --year_start_test 2020 \
    --year_end_test 2020 \
    --batch_size 70 \
    --num_epochs 1 \
    --learning_rate 0.0001 \
    --num_workers 8 \
    --datadir /leonardo_work/EUHPC_D27_095/kkingston/AI-Downscaling/data/data_FOURxDaily \
    --per_var_datadir VAR_2T=/leonardo_work/EUHPC_D27_095/kkingston/AI-Downscaling/data/data_FOURxDaily \
    --time_normalization cos_sin \
    --tbatch 1800 \
    --sbatch 12 \
    --batch_size_lat 145 \
    --batch_size_lon 361 \
    --epsilon 0.02 \
    --beta 1.0 \
    --margin 8 \
    --dynamic_covariates_dir ../data_covariates/ \
    --dtype fp32 \
    --num_steps 10 \
    --sigma_min 0.002 \
    --sigma_max 80.0 \
    --rho 7 \
    --s_churn 40 \
    --solver heun \
    --compute_crps false
"""

In [3]:
# Use bash -c to ensure all commands (export, cd, python) run in the same environment
!bash -c "{cmd}"

╭────────────────────────── IPSL AI Downscaling Tool ──────────────────────────╮
│ 🚀 Starting Module: Main                                                     │
╰──────────────────────────────────────────────────────────────────────────────╯
[INFO] ============================================================
[INFO] CONFIGURATION PARAMETERS
[INFO] ============================================================
[INFO] Execution Mode:
[INFO]  └── Debug: False
[INFO]  └── Run type: 'train_regional'
[INFO]  └── Inference type: 'sampler'
[INFO]  └── Region: 'europe'
[INFO]  └── Apply filter: False
[INFO] 
Checkpoint Configuration:
[INFO]  └── Save model: True
[INFO]  └── Save checkpoint name: 'difusion_model'
[INFO]  └── Load checkpoint name: 'difusion_model'
[INFO]  └── Save per samples: 10000
[INFO]  └── Model name: 'notebook_test_run_diffusion_Europe'
[INFO] 
Data Configuration:
[INFO]  └── Variable names: ['VAR_2T']
[INFO]  └── Constant variables: ['z', 'lsm']
[INFO]  └── Constant variables

## U-Net Model - Training Command

This example runs a **simple training configuration on a selected region**, with `--run_type` set to `train_regional`:
- **1 year of training data** (2019)
- **1 year of validation data** (2020)
- **1 epoch only**

It is intended for quick testing and validation of the pipeline.

### Outputs

All outputs are saved in the following directories:

- **Results (plots, diagnostics):**  
  `results/notebook_test/test_run_unet`

- **Logs:**  
  `logs/notebook_test/test_run_unet`

- **Checkpoints:**  
  `checkpoints/notebook_test/test_run_unet`

In [4]:
cmd = """

cd "/leonardo_work/EUHPC_D27_095/kkingston/IPSL-AID"
source .venv/bin/activate

ipsl-aid \
    --debug false \
    --main_folder notebook_test \
    --sub_folder test_run_unet_Europe \
    --prefix run_20260323_142528 \
    --run_type train_regional \
    --region "Europe" \
    --region_size 288 720 \
    --save_model true \
    --save_checkpoint_name unet_model \
    --load_checkpoint_name unet_model \
    --save_per_samples 10000 \
    --inference_type direct \
    --arch adm \
    --precond unet \
    --in_channels 5 \
    --out_channels 1 \
    --apply_filter false \
    --model_name notebook_test_run_unet_Europe \
    --varnames_list VAR_2T \
    --normalization_types VAR_2T=standard \
    --constant_varnames_list z lsm \
    --constant_varnames_file ERA5_const_sfc_variables.nc \
    --units_list K \
    --year_start 2019 \
    --year_end 2019 \
    --year_start_test 2020 \
    --year_end_test 2020 \
    --batch_size 70 \
    --num_epochs 1 \
    --learning_rate 0.0001 \
    --num_workers 8 \
    --datadir /leonardo_work/EUHPC_D27_095/kkingston/AI-Downscaling/data/data_FOURxDaily \
    --per_var_datadir VAR_2T=/leonardo_work/EUHPC_D27_095/kkingston/AI-Downscaling/data/data_FOURxDaily \
    --time_normalization cos_sin \
    --tbatch 1800 \
    --sbatch 12 \
    --batch_size_lat 145 \
    --batch_size_lon 361 \
    --epsilon 0.02 \
    --beta 1.0 \
    --margin 8 \
    --dynamic_covariates_dir ../data_covariates/ \
    --dtype fp32
"""

In [5]:
# Use bash -c to ensure all commands (export, cd, python) run in the same environment
!bash -c "{cmd}"

╭────────────────────────── IPSL AI Downscaling Tool ──────────────────────────╮
│ 🚀 Starting Module: Main                                                     │
╰──────────────────────────────────────────────────────────────────────────────╯
[INFO] ============================================================
[INFO] CONFIGURATION PARAMETERS
[INFO] ============================================================
[INFO] Execution Mode:
[INFO]  └── Debug: False
[INFO]  └── Run type: 'train_regional'
[INFO]  └── Inference type: 'direct'
[INFO]  └── Region: 'europe'
[INFO]  └── Apply filter: False
[INFO] 
Checkpoint Configuration:
[INFO]  └── Save model: True
[INFO]  └── Save checkpoint name: 'unet_model'
[INFO]  └── Load checkpoint name: 'unet_model'
[INFO]  └── Save per samples: 10000
[INFO]  └── Model name: 'notebook_test_run_unet_Europe'
[INFO] 
Data Configuration:
[INFO]  └── Variable names: ['VAR_2T']
[INFO]  └── Constant variables: ['z', 'lsm']
[INFO]  └── Constant variables file: 'ERA5_c